# 00 — SPLADE document encoding for MS MARCO passage (Kaggle GPU)

Encodes all 8,841,823 passages with a SPLADE model and writes compact sparse shards for the
learned-sparse arms (A3/A4). Run on Kaggle with **GPU T4 x2** and **Internet on**, via
*Save Version -> Save & Run All* so it runs unattended. Output goes to `/kaggle/working/splade/`.

* **Resumable:** shards that already exist are skipped. If a session hits the 12 h limit, attach
  the previous version's output as input, set `RESUME_FROM` below, and run again.
* **Shard format** (`shard_XXXX.npz`): CSR matrix: `indptr` int64, `indices` uint16 (BERT
  vocab ids < 30522), `values` float16, plus `doc_ids`.
* **Model:** `naver/splade-cocondenser-ensembledistil` (not gated, and the model used in the
  Seismic paper). To use `naver/splade-v3` instead, accept its licence on Hugging Face and add an
  `HF_TOKEN` Kaggle secret.

In [ ]:
!pip install -q ir_datasets

In [ ]:
import json, os, shutil, time
from pathlib import Path

import ir_datasets
import numpy as np
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

MODEL = "naver/splade-cocondenser-ensembledistil"
MAX_LEN = 256
BATCH = 128            # per forward pass, split across GPUs
SHARD_SIZE = 250_000   # passages per output shard (36 shards)
OUT = Path("/kaggle/working/splade")
RESUME_FROM = None     # e.g. Path("/kaggle/input/<previous-version-output>/splade")
OUT.mkdir(parents=True, exist_ok=True)
if RESUME_FROM is not None:
    for f in RESUME_FROM.glob("shard_*.npz"):
        if not (OUT / f.name).exists():
            shutil.copy(f, OUT / f.name)
print(torch.cuda.device_count(), "GPU(s)", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
class SpladeDocEncoder(torch.nn.Module):
    """Returns the max-pooled SPLADE vector, so DataParallel gathers (B, V), not (B, L, V)."""

    def __init__(self, name):
        super().__init__()
        self.mlm = AutoModelForMaskedLM.from_pretrained(name)

    def forward(self, input_ids, attention_mask):
        logits = self.mlm(input_ids=input_ids, attention_mask=attention_mask).logits
        weights = torch.log1p(torch.relu(logits)) * attention_mask.unsqueeze(-1)
        return weights.max(dim=1).values

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = SpladeDocEncoder(MODEL).half().cuda().eval()
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
assert tokenizer.vocab_size <= np.iinfo(np.uint16).max

In [ ]:
dataset = ir_datasets.load("msmarco-passage")
n_docs = dataset.docs_count()
n_shards = (n_docs + SHARD_SIZE - 1) // SHARD_SIZE
print(n_docs, "passages ->", n_shards, "shards")

In [ ]:
@torch.inference_mode()
def encode_shard(texts):
    # Sort by length so batches are padded as little as possible, then restore order.
    order = np.argsort([len(t) for t in texts])
    rows = [None] * len(texts)
    for start in range(0, len(texts), BATCH):
        idx = order[start:start + BATCH]
        enc = tokenizer([texts[i] for i in idx], max_length=MAX_LEN, truncation=True,
                        padding=True, return_tensors="pt").to("cuda")
        vecs = model(enc["input_ids"], enc["attention_mask"]).float().cpu()
        for row, i in zip(vecs, idx):
            nz = torch.nonzero(row, as_tuple=True)[0]
            rows[i] = (nz.numpy().astype(np.uint16), row[nz].numpy().astype(np.float16))
    indptr = np.zeros(len(rows) + 1, dtype=np.int64)
    indptr[1:] = np.cumsum([len(r[0]) for r in rows])
    return indptr, np.concatenate([r[0] for r in rows]), np.concatenate([r[1] for r in rows])

log = []
docs = iter(dataset.docs_iter())
for s in range(n_shards):
    path = OUT / f"shard_{s:04d}.npz"
    batch = [next(docs) for _ in range(min(SHARD_SIZE, n_docs - s * SHARD_SIZE))]
    if path.exists():
        continue
    t0 = time.time()
    indptr, indices, values = encode_shard([d.text for d in batch])
    np.savez(path, doc_ids=np.array([d.doc_id for d in batch]), indptr=indptr, indices=indices, values=values)
    log.append({"shard": s, "docs": len(batch), "seconds": time.time() - t0, "avg_nnz": float(indptr[-1] / len(batch))})
    print(log[-1], flush=True)

In [ ]:
meta = {"model": MODEL, "max_len": MAX_LEN, "n_docs": n_docs, "shard_size": SHARD_SIZE,
        "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], "log": log}
(OUT / "meta.json").write_text(json.dumps(meta, indent=2))
print(sum(f.stat().st_size for f in OUT.glob("*.npz")) / 1e9, "GB in", len(list(OUT.glob("*.npz"))), "shards")